# Notebook 1 — Data Collection from Google Earth Engine

This notebook contains all the Google Earth Engine (GEE) JavaScript scripts used to collect data for the Dhemaji flood prediction project.

## Data Sources
1. Sentinel-1 SAR — Flood labels (2019-2024)
2. CHIRPS — Local rainfall (2019-2024)
3. CHIRPS — Upstream catchment rainfall (Arunachal Pradesh)
4. ERA5 Land — River runoff (proxy for water level)
5. HydroSHEDS — River network for distance calculation

## Instructions
1. Open https://code.earthengine.google.com
2. Copy each JavaScript block below into a new GEE script
3. Run the script and check console output
4. Go to Tasks tab and click Run on the export task
5. Wait for export to complete (5-60 minutes depending on size)
6. Download CSV/SHP from Google Drive folder `flood_project`

## Script 1 — Sentinel-1 SAR Flood Labels (Multi-year)

Extracts dynamic flood labels from Sentinel-1 SAR backscatter data.
Water surfaces appear dark in SAR (VV < -15 dB).
Covers monsoon seasons June-September from 2019 to 2024.

In [ ]:
%%javascript
// ============================================
// DHEMAJI FLOOD LABEL EXTRACTION
// Sentinel-1 SAR — Monsoon Seasons 2019-2024
// ============================================

var dhemaji = ee.Geometry.Rectangle([
    94.35, 27.40,
    94.85, 27.75
]);

// Load Sentinel-1 for all monsoon seasons
var s1 = ee.ImageCollection("COPERNICUS/S1_GRD")
  .filterBounds(dhemaji)
  .filter(ee.Filter.or(
    ee.Filter.date("2019-06-01", "2019-09-30"),
    ee.Filter.date("2020-06-01", "2020-09-30"),
    ee.Filter.date("2021-06-01", "2021-09-30"),
    ee.Filter.date("2022-06-01", "2022-09-30"),
    ee.Filter.date("2023-06-01", "2023-09-30"),
    ee.Filter.date("2024-06-01", "2024-09-30")
  ))
  .filter(ee.Filter.eq("instrumentMode", "IW"))
  .filter(ee.Filter.listContains(
      "transmitterReceiverPolarisation", "VV"))
  .select("VV");

print("Total images found:", s1.size());

// Flood detection — VV backscatter < -15dB = water
var detectFlood = function(image) {
  var date = image.date().format("dd-MM-YYYY");
  var flood = image.lt(-15)
    .rename("flood_label")
    .set("date", date);
  return flood;
};

var floodCollection = s1.map(detectFlood);

// Sample at 500m grid
var exportFloodImage = function(image) {
  var date = image.getString("date");
  var samples = image.addBands(
    ee.Image.pixelLonLat()
  ).sample({
    region: dhemaji,
    scale: 500,
    projection: "EPSG:4326",
    geometries: true
  });
  samples = samples.map(function(f) {
    return f.set("date", date);
  });
  return samples;
};

var allDates = floodCollection.toList(floodCollection.size());
var merged = ee.FeatureCollection(
  allDates.map(function(img) {
    return exportFloodImage(ee.Image(img));
  })
).flatten();

print("Total labeled points:", merged.size());

Export.table.toDrive({
  collection: merged,
  description: "DHEMAJI_FLOOD_LABELS_MULTIYEAR",
  folder: "flood_project",
  fileNamePrefix: "dhemaji_flood_labels_multiyear",
  fileFormat: "CSV"
});

## Script 2 — Local Rainfall from CHIRPS

CHIRPS daily precipitation data at 5km resolution, sampled at 500m grid for Dhemaji.

In [ ]:
%%javascript
// ============================================
// LOCAL RAINFALL EXTRACTION — CHIRPS
// ============================================

var dhemaji = ee.Geometry.Rectangle([
    94.35, 27.40,
    94.85, 27.75
]);

var chirps_local = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
  .filter(ee.Filter.or(
    ee.Filter.date("2019-05-20", "2019-09-30"),
    ee.Filter.date("2020-05-20", "2020-09-30"),
    ee.Filter.date("2021-05-20", "2021-09-30"),
    ee.Filter.date("2022-05-20", "2022-09-30"),
    ee.Filter.date("2023-05-20", "2023-09-30"),
    ee.Filter.date("2024-05-20", "2024-09-30")
  ))
  .filterBounds(dhemaji)
  .select("precipitation");

print("Local rainfall images:", chirps_local.size());

var merged_local = ee.FeatureCollection(
  chirps_local.toList(chirps_local.size()).map(function(img) {
    var image = ee.Image(img);
    var date = image.date().format("dd-MM-YYYY");
    var samples = image.rename("rainfall_mm")
      .addBands(ee.Image.pixelLonLat())
      .sample({
        region: dhemaji,
        scale: 500,
        projection: "EPSG:4326",
        geometries: true
      });
    return samples.map(function(f) {
      return f.set("date", date);
    });
  })
).flatten();

Export.table.toDrive({
  collection: merged_local,
  description: "LOCAL_RAINFALL_MULTIYEAR",
  folder: "flood_project",
  fileNamePrefix: "local_rainfall_multiyear",
  fileFormat: "CSV"
});

## Script 3 — Upstream Catchment Rainfall

Rainfall in the Arunachal Pradesh hills that drain into Brahmaputra.
This captures the delayed flood effect from upstream rainfall.

In [ ]:
%%javascript
// ============================================
// UPSTREAM CATCHMENT RAINFALL
// Arunachal Pradesh — Brahmaputra upstream
// ============================================

var upstream = ee.Geometry.Rectangle([
    93.0, 27.80,
    96.0, 29.50
]);

var chirps_upstream = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
  .filter(ee.Filter.or(
    ee.Filter.date("2019-05-25", "2019-09-30"),
    ee.Filter.date("2020-05-25", "2020-09-30"),
    ee.Filter.date("2021-05-25", "2021-09-30"),
    ee.Filter.date("2022-05-25", "2022-09-30"),
    ee.Filter.date("2023-05-25", "2023-09-30"),
    ee.Filter.date("2024-05-25", "2024-09-30")
  ))
  .filterBounds(upstream)
  .select("precipitation");

var upstream_daily = chirps_upstream.map(function(img) {
  var mean_rain = img.reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: upstream,
    scale: 5000
  });
  return ee.Feature(null, {
    "date": img.date().format("dd-MM-YYYY"),
    "upstream_rainfall": mean_rain.get("precipitation")
  });
});

Export.table.toDrive({
  collection: ee.FeatureCollection(upstream_daily),
  description: "UPSTREAM_RAINFALL_MULTIYEAR",
  folder: "flood_project",
  fileNamePrefix: "upstream_rainfall_multiyear",
  fileFormat: "CSV"
});

## Script 4 — ERA5 River Runoff

ERA5 Land surface runoff data as proxy for river water level.
Used when actual gauge data from CWC was unavailable.

In [ ]:
%%javascript
// ============================================
// BRAHMAPUTRA RIVER RUNOFF — ERA5
// ============================================

var brahmaputra_corridor = ee.Geometry.Rectangle([
    94.35, 27.40,
    95.10, 27.75
]);

var era5 = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
  .filter(ee.Filter.or(
    ee.Filter.date("2019-06-01", "2019-09-30"),
    ee.Filter.date("2020-06-01", "2020-09-30"),
    ee.Filter.date("2021-06-01", "2021-09-30"),
    ee.Filter.date("2022-06-01", "2022-09-30"),
    ee.Filter.date("2023-06-01", "2023-09-30"),
    ee.Filter.date("2024-06-01", "2024-09-30")
  ))
  .select("runoff_sum");

var runoff_daily = era5.map(function(img) {
  var runoff = img.reduceRegion({
    reducer: ee.Reducer.mean(),
    geometry: brahmaputra_corridor,
    scale: 11132
  });
  return ee.Feature(null, {
    "date": img.date().format("dd-MM-YYYY"),
    "runoff_sum": runoff.get("runoff_sum")
  });
});

Export.table.toDrive({
  collection: ee.FeatureCollection(runoff_daily),
  description: "BRAHMAPUTRA_RUNOFF",
  folder: "flood_project",
  fileNamePrefix: "brahmaputra_runoff",
  fileFormat: "CSV"
});

## Script 5 — River Network (HydroSHEDS)

HydroSHEDS river network data for computing distance to major rivers.
Filter to major rivers (RIV_ORD <= 4) for Brahmaputra and main tributaries.

In [ ]:
%%javascript
// ============================================
// RIVER NETWORK EXTRACTION — HydroSHEDS
// ============================================

var dhemaji = ee.Geometry.Rectangle([
    94.35, 27.40,
    94.85, 27.75
]);

var rivers = ee.FeatureCollection(
    "WWF/HydroSHEDS/v1/FreeFlowingRivers"
)
.filterBounds(dhemaji);

print("Total river segments:", rivers.size());

// Visualize
Map.centerObject(dhemaji, 10);
Map.addLayer(dhemaji, {color: "red"}, "Dhemaji");
Map.addLayer(rivers, {color: "blue"}, "Rivers");

Export.table.toDrive({
  collection: rivers,
  description: "DHEMAJI_RIVERS",
  folder: "flood_project",
  fileNamePrefix: "dhemaji_rivers",
  fileFormat: "SHP"
});